# Module 2 — Take-Home Exercises: Fine-Tuning Deep Dive

These exercises reinforce what you learned about QLoRA fine-tuning,
hyperparameter choices, and interpreting training results.

**Exercise 1-2:** No GPU needed (analysis only)
**Exercise 3-4:** T4 GPU required (Colab free tier)

---

## Exercise 1: Hyperparameter Impact Analysis (No GPU)

In class, we used these two configurations:

| Parameter | v1 (ChatDoctor) | v2 (WikiDoc) |
|---|---|---|
| learning_rate | 2e-4 | 5e-5 |
| num_train_epochs | 3 | 2 |
| warmup_steps | 75 | 50 |
| dataset_size | 2,000 | 2,000 |

**Tasks:**

1. Calculate the total training steps for each experiment
   (assume effective batch size = 8, so steps = dataset_size / batch_size × epochs)
2. What percentage of total steps is the warmup period for each?
3. If you doubled the learning rate for v2 (from 5e-5 to 1e-4), predict what happens.
   Would accuracy improve or degrade? Would helpfulness change? Explain your reasoning.
4. If you trained v2 for 5 epochs instead of 2, what risk increases? What metric
   from Module 4 would you watch most carefully?

In [ ]:
# TODO: Calculate training steps

# v1
v1_steps_per_epoch = 2000 // 8  # dataset_size / effective_batch_size
v1_total_steps = None  # TODO: multiply by epochs
v1_warmup_pct = None   # TODO: warmup_steps / total_steps * 100

# v2
v2_steps_per_epoch = 2000 // 8
v2_total_steps = None  # TODO
v2_warmup_pct = None   # TODO

print(f"v1: {v1_total_steps} total steps, {v1_warmup_pct:.1f}% warmup")
print(f"v2: {v2_total_steps} total steps, {v2_warmup_pct:.1f}% warmup")

**Your Predictions (fill in):**

3. Doubling v2 learning rate (5e-5 → 1e-4):
   - Accuracy would: (improve / stay same / degrade) because...
   - Helpfulness would: (improve / stay same / degrade) because...

4. Training v2 for 5 epochs:
   - The main risk is: ...
   - The metric to watch is: ...

---

## Exercise 2: Analyze Benchmark Results (No GPU)

Load the v1 and v2 benchmark results from class and analyze them programmatically.

**Tasks:**

1. Load both JSON files and extract the 10 question-answer pairs
2. Compare average response LENGTH (characters) between base and fine-tuned for each version
3. For v2: identify which questions got SHORTER after fine-tuning and by how much
4. Is there a correlation between answer length and the helpfulness scores from Module 4?

In [ ]:
import json
import pandas as pd

# Load benchmark results
# NOTE: Update paths if running on Colab (upload files first)
with open("../results/benchmark_results_v1.json") as f:
    v1_results = json.load(f)

with open("../results/benchmark_results_v2.json") as f:
    v2_results = json.load(f)

print(f"v1: {len(v1_results)} results")
print(f"v2: {len(v2_results)} results")
print(f"\nKeys in each result: {list(v1_results[0].keys())}")

In [ ]:
# TODO: Compare average response length

for label, results in [("v1", v1_results), ("v2", v2_results)]:
    base_lengths = [len(r["base_output"]) for r in results]
    ft_lengths = [len(r["finetuned_output"]) for r in results]
    
    avg_base = None  # TODO: calculate average
    avg_ft = None    # TODO: calculate average
    
    print(f"\n{label}:")
    print(f"  Base model avg length: {avg_base:.0f} chars")
    print(f"  Fine-tuned avg length: {avg_ft:.0f} chars")
    print(f"  Change: {avg_ft - avg_base:+.0f} chars ({(avg_ft/avg_base - 1)*100:+.1f}%)")

In [ ]:
# TODO: For v2, find which questions got shorter

print("v2: Questions that got SHORTER after fine-tuning:")
print(f"{'Question':<55} {'Base':>6} {'FT':>6} {'Delta':>7}")
print("-" * 80)

for r in v2_results:
    base_len = len(r["base_output"])
    ft_len = len(r["finetuned_output"])
    delta = ft_len - base_len
    
    # TODO: Only print questions where delta < 0 (got shorter)
    pass

**Your Findings (fill in):**

1. Which version (v1 or v2) changed response length more dramatically?
2. Does the length change correlate with the helpfulness regression we saw in Module 4?
3. What does this suggest about the training data's influence on output style?

---

## Exercise 3: LoRA Rank Experiment (T4 GPU Required)

In class, we used LoRA rank r=16. Higher rank = more trainable parameters = more
capacity to learn, but also more risk of overfitting.

**Tasks:**

1. Fine-tune the v2 model (WikiDoc) with r=4 (very low rank) instead of r=16
2. Compare the training loss curves: does r=4 converge slower?
3. Run the same 10 benchmark prompts and compare outputs to the r=16 version
4. Calculate: how many trainable parameters does r=4 have vs r=16?

In [ ]:
!pip install -q transformers torch bitsandbytes accelerate peft trl datasets

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")

In [ ]:
# TODO: Create LoRA config with r=4 (instead of r=16 from class)

lora_config = LoraConfig(
    r=4,             # <-- Changed from 16
    lora_alpha=8,    # <-- Keep alpha = 2*r  (was 32 for r=16)
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

# TODO: Note the trainable parameter count here
# Compare to r=16 (which had ~14M trainable params)
# Question: Is it roughly 4/16 = 25% of the r=16 count?

In [ ]:
# Load dataset (same as v2 from class)
dataset = load_dataset("jeev1992/wikidoc-healthassist", split="train")
dataset = dataset.shuffle(seed=42)
train_dataset = dataset.select(range(2000))
eval_dataset = dataset.select(range(2000, 2100))

# TODO: Set up SFTConfig and SFTTrainer (same hyperparams as v2 from class)
# training_args = SFTConfig(
#     output_dir="./lora-r4-experiment",
#     num_train_epochs=2,
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=2,
#     learning_rate=5e-5,
#     warmup_steps=50,
#     ...
# )

# TODO: Train and compare loss to the r=16 run

**Your Findings (fill in):**

1. Trainable parameters: r=4 has ___ params vs r=16 had ~14M
2. Final training loss: r=4 = ___ vs r=16 = ___
3. Output quality comparison (pick 2 benchmark questions and compare):
   - r=4 outputs were: (better / similar / worse) because...
4. For a 1.5B model with 2,000 training examples, is r=4 sufficient? Why or why not?

---

## Exercise 4: Training Data Ablation (T4 GPU Required)

We trained on 2,000 examples. What happens with fewer?

**Tasks:**

1. Train on only **200 examples** (10% of original) with the same v2 hyperparams
2. Train on **500 examples** (25% of original)
3. Compare the 3 models (200, 500, 2000) on 3 benchmark questions
4. At what point do you see diminishing returns?

In [ ]:
# TODO: Reuse the setup from Exercise 3, but vary the dataset size

# Experiment 1: 200 examples
# train_200 = dataset.select(range(200))
# ... train, benchmark, record outputs

# Experiment 2: 500 examples  
# train_500 = dataset.select(range(500))
# ... train, benchmark, record outputs

# Compare outputs on these 3 questions:
test_questions = [
    "What are the early warning signs of a stroke?",
    "How does metformin work for diabetes management?",
    "What lifestyle changes help manage high cholesterol?",
]

# HINT: To save Colab time, you can train each in sequence.
# Remember to delete the model and clear cache between runs:
#   del peft_model
#   torch.cuda.empty_cache()
#   model = AutoModelForCausalLM.from_pretrained(...)

**Your Findings (fill in):**

1. 200 examples: Safety disclaimers present? (yes/no). Output quality: (1-5)
2. 500 examples: Safety disclaimers present? (yes/no). Output quality: (1-5)
3. 2000 examples (from class): Safety disclaimers present? (yes). Output quality: ___
4. Diminishing returns start at approximately ___ examples because...
5. What's the minimum dataset size you'd recommend for this task? Why?